## Scenario: Chronic Pain Only Claimants

**Description:** An active claim whose diagnoses are *only* chronic pain (ICD-9 338.2x/338.4x or ICD-10 G89.2x/G89.4x) with no other diagnosis recorded. Chronic pain alone rarely qualifies for LTC benefits, so a claim carrying no other qualifying diagnosis warrants review.

In [ ]:
ENGINE_CATALOG = dbutils.widgets.get("ENGINE_CATALOG")
ENGINE_SCHEMA = dbutils.widgets.get("ENGINE_SCHEMA")

GRAPH_CATALOG = dbutils.widgets.get("GRAPH_CATALOG")
GRAPH_SCHEMA = dbutils.widgets.get("GRAPH_SCHEMA")

In [ ]:
%sql

DECLARE execDatetime TIMESTAMP = GETDATE();

In [ ]:
# Scenario is not yet registered in T_NOVEL_SCENARIO; skip during testing.
# df = spark.sql(f"""
#   SELECT 
#     NOVEL_SCENARIO_ID 
#   FROM
#     {ENGINE_CATALOG}.{ENGINE_SCHEMA}.T_NOVEL_SCENARIO 
#   WHERE 
#     NOTEBOOK_NAME = 'Scenario_ChronicPainOnlyCaimants'
# """)
#
# novelScenarioId = df.collect()[0][0]
# print(f"Novel scenario ID: {novelScenarioId}")

## Parameters

In [ ]:
CHRONIC_PAIN_ICD9_PREFIXES = ["338.2", "338.4"]
CHRONIC_PAIN_ICD10_PREFIXES = ["G89.2", "G89.4"]
ACTIVE_CLAIM_STATUSES = ["Active", "ASWP", "Benefit Period", "Qualification Period"]

active_statuses = ", ".join(f"'{s}'" for s in ACTIVE_CLAIM_STATUSES)
chronic_pain_condition = " OR ".join(
    [f"ICD9_CODE LIKE '{p}%'" for p in CHRONIC_PAIN_ICD9_PREFIXES]
    + [f"ICD10_CODE LIKE '{p}%'" for p in CHRONIC_PAIN_ICD10_PREFIXES]
)
print(chronic_pain_condition)

## Claim diagnosis rollup

In [ ]:
spark.sql(f"""
SELECT
  CLAIM_ID,
  MAX(CASE WHEN {chronic_pain_condition} THEN 1 ELSE 0 END) AS HAS_CHRONIC_PAIN,
  MAX(CASE WHEN NOT ({chronic_pain_condition}) THEN 1 ELSE 0 END) AS HAS_OTHER_DX,
  MAX(CASE WHEN ICD9_CODE LIKE '338.2%' OR ICD9_CODE LIKE '338.4%' THEN ICD9_CODE END) AS CHRONIC_PAIN_ICD9_CODE,
  MAX(CASE WHEN ICD10_CODE LIKE 'G89.2%' OR ICD10_CODE LIKE 'G89.4%' THEN ICD10_CODE END) AS CHRONIC_PAIN_ICD10_CODE,
  SUM(CASE WHEN {chronic_pain_condition} THEN 1 ELSE 0 END) AS CHRONIC_PAIN_DX_COUNT
FROM {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_NORM_DIAGNOSIS
WHERE CLAIM_ID IS NOT NULL
GROUP BY CLAIM_ID
""").createOrReplaceTempView("claim_dx")

## Claims with chronic pain and no other diagnosis

In [ ]:
spark.sql("""
SELECT
  CLAIM_ID,
  CHRONIC_PAIN_ICD9_CODE,
  CHRONIC_PAIN_ICD10_CODE,
  CHRONIC_PAIN_DX_COUNT
FROM claim_dx
WHERE HAS_CHRONIC_PAIN = 1
  AND HAS_OTHER_DX = 0
""").createOrReplaceTempView("chronic_pain_only_claims")

## Invoice totals per claim

In [ ]:
spark.sql(f"""
SELECT
  CLAIM_ID,
  SUM(INVOICE_CHARGE_AMT) AS TOTAL_CHARGE_AMT,
  SUM(INVOICE_PAY_AMT) AS TOTAL_PAY_AMT
FROM {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_NORM_INVOICE
GROUP BY CLAIM_ID
""").createOrReplaceTempView("claim_invoice_totals")

## Flagged claims

In [ ]:
spark.sql(f"""
SELECT
  CLAIM_ID,
  CLAIM_NUMBER,
  CLAIMANT_ID,
  CLAIMANT_FIRST_NAME,
  CLAIMANT_LAST_NAME,
  CLAIMANT_BIRTH_DATE,
  CLAIM_STATUS_CODE,
  CLAIM_OPEN_DATE,
  CHRONIC_PAIN_ICD9_CODE,
  CHRONIC_PAIN_ICD10_CODE,
  CHRONIC_PAIN_DX_COUNT,
  TOTAL_CHARGE_AMT,
  TOTAL_PAY_AMT
FROM (
  SELECT
    cl.CLAIM_ID,
    cl.CLAIM_NUMBER,
    rp.RES_PERSON_ID AS CLAIMANT_ID,
    rp.FIRST_NAME AS CLAIMANT_FIRST_NAME,
    rp.LAST_NAME AS CLAIMANT_LAST_NAME,
    rp.BIRTH_DATE AS CLAIMANT_BIRTH_DATE,
    cl.CLAIM_STATUS_CODE,
    cl.CLAIM_OPEN_DATE,
    cpo.CHRONIC_PAIN_ICD9_CODE,
    cpo.CHRONIC_PAIN_ICD10_CODE,
    cpo.CHRONIC_PAIN_DX_COUNT,
    COALESCE(it.TOTAL_CHARGE_AMT, 0) AS TOTAL_CHARGE_AMT,
    COALESCE(it.TOTAL_PAY_AMT, 0) AS TOTAL_PAY_AMT,
    ROW_NUMBER() OVER (
      PARTITION BY cl.CLAIM_NUMBER
      ORDER BY cl.CLAIM_OPEN_DATE DESC, cl.CLAIM_ID
    ) AS RN
  FROM chronic_pain_only_claims cpo
  JOIN {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_NORM_CLAIM cl
    ON cl.CLAIM_ID = cpo.CLAIM_ID
  JOIN {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_RESOLVED_PERSON_POLICY_CROSSWALK rppc
    ON rppc.POLICY_NUMBER = cl.POLICY_NUMBER
    AND rppc.EDGE_NAME = 'IS_COVERED_BY'
  JOIN {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_RESOLVED_PERSON rp
    ON rp.RES_PERSON_ID = rppc.RES_PERSON_ID
    AND UPPER(TRIM(cl.FIRST_NAME)) = UPPER(TRIM(rp.FIRST_NAME))
    AND UPPER(TRIM(cl.LAST_NAME)) = UPPER(TRIM(rp.LAST_NAME))
  LEFT JOIN claim_invoice_totals it
    ON it.CLAIM_ID = cl.CLAIM_ID
  WHERE cl.CLAIM_STATUS_CODE IN ({active_statuses})
)
WHERE RN = 1
""").createOrReplaceTempView("flagged_claims")

## Trigger counts

In [ ]:
flagged = spark.table("flagged_claims").cache()

n_rows      = flagged.count()
n_claims    = flagged.select("CLAIM_ID").distinct().count()
n_claimants = flagged.select("CLAIMANT_ID").distinct().count()

print(f"Flagged rows            : {n_rows:,}")
print(f"Distinct trigger claims : {n_claims:,}")
print(f"Distinct claimants      : {n_claimants:,}")

## Preview / browse full result

In [ ]:
# Use the Databricks table UI to sort/filter and click 'Download' for CSV/Excel.
display(flagged.orderBy("CLAIM_OPEN_DATE", ascending=False))

In [ ]:
# Engine table does not exist yet; skip during testing.
# spark.sql(f"""
#     INSERT INTO {ENGINE_CATALOG}.{ENGINE_SCHEMA}.T_SCENARIO_CHRONIC_PAIN_ONLY_CLAIMANTS_DETAIL
#     SELECT 
#       *,
#       GETDATE() AS FEATURE_DATETIME
#     FROM flagged_claims;
# """)